In [ ]:
#ORIGINAL VIDEO PROCESSING SCRIPT
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from imageio import get_reader
import matplotlib.pyplot as plt
import numpy as np

PATCH_SIZE = 32
RESIZE_TO = (256, 256)
DEVICE = torch.device("cuda")
BATCH_SIZE = 1  # one video at a time (frames are parallelized inside)
OUTPUT_DIR = ".output3/veo3_spectra_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Gaussian kernel
def get_gaussian_kernel(kernel_size=5, sigma=1.0, device='cuda'):
    ax = torch.arange(-kernel_size // 2 + 1., kernel_size // 2 + 1., device=device)
    xx, yy = torch.meshgrid(ax, ax, indexing='xy')
    kernel = torch.exp(-(xx**2 + yy**2) / (2. * sigma**2))
    kernel = kernel / kernel.sum()
    return kernel.view(1, 1, kernel_size, kernel_size) 

def fftshift2d(x):
    shift_y = x.size(-2) // 2
    shift_x = x.size(-1) // 2
    return torch.roll(torch.roll(x, shifts=shift_y, dims=-2), shifts=shift_x, dims=-1)

# Dataset class for loading videos 
class VideoDataset(Dataset):
    def __init__(self, video_dir, transform=None, frame_limit=None):
        self.video_paths = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.endswith(".mp4")]
        self.transform = transform
        self.frame_limit = frame_limit

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        reader = get_reader(self.video_paths[idx])
        frames = []
        count = 0
        for frame in reader:
            img = Image.fromarray(frame).convert("L")
            if self.transform:
                img = self.transform(img)
            frames.append(img)
            count += 1
            if self.frame_limit and count >= self.frame_limit:
                break
        reader.close()
        return torch.stack(frames), os.path.basename(self.video_paths[idx])

# Transform
transform = transforms.Compose([
    transforms.Resize(RESIZE_TO),
    transforms.ToTensor() 
])

# Initialize dataset and loader 
video_dir = "./AI_Video_Generation_Dataset/veo3"
dataset = VideoDataset(video_dir, transform=transform, frame_limit=100)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Precomputed Gaussian kernel on device 
gaussian_kernel = get_gaussian_kernel(kernel_size=5, sigma=1.0, device=DEVICE)

# Main loop over videos
for batch in dataloader:
    video_frames, video_name = batch
    video_frames = video_frames.squeeze(0).to(DEVICE) 
    video_name = video_name[0].replace('.mp4', '')

    # Split frames into non-overlapping patches using reshape 
    T, C, H, W = video_frames.shape
    assert H % PATCH_SIZE == 0 and W % PATCH_SIZE == 0, "Image size must be divisible by patch size"

    patches = video_frames.reshape(T, C, H // PATCH_SIZE, PATCH_SIZE, W // PATCH_SIZE, PATCH_SIZE)
    patches = patches.permute(0, 2, 4, 1, 3, 5).reshape(-1, 1, PATCH_SIZE, PATCH_SIZE)  # (N_patches_total, 1, PATCH_SIZE, PATCH_SIZE)

    # Apply filter 
    blurred = F.conv2d(patches, gaussian_kernel, padding=2, groups=1)
    
    # High pass 
    filtered_patches = patches - blurred  # (N_patches_total, 1, PATCH_SIZE, PATCH_SIZE)
    filtered_patches = filtered_patches - filtered_patches.mean(dim=(-2, -1), keepdim=True)
    # FFT
    fft = torch.fft.fft2(filtered_patches)
    fft_shifted = torch.fft.fftshift(fft)
    magnitude = torch.log1p(torch.abs(fft_shifted))  # (N_patches, 1, PATCH_SIZE, PATCH_SIZE)

    # Mean spectrum over all patches
    avg_spectrum = magnitude.mean(dim=0).squeeze().detach().cpu()  # shape: (PATCH_SIZE, PATCH_SIZE)

    # Save as PNG 
    output_path = os.path.join(OUTPUT_DIR, f"{video_name}.png")
    plt.imsave(output_path, avg_spectrum.numpy(), cmap='gray')
    print(f"Saved: {output_path}")

Saved: .output3/veo3_spectra_output/1.png
Saved: .output3/veo3_spectra_output/14.png
Saved: .output3/veo3_spectra_output/9.png
Saved: .output3/veo3_spectra_output/15.png
Saved: .output3/veo3_spectra_output/13.png
Saved: .output3/veo3_spectra_output/24.png
Saved: .output3/veo3_spectra_output/10.png
Saved: .output3/veo3_spectra_output/18.png
Saved: .output3/veo3_spectra_output/19.png
Saved: .output3/veo3_spectra_output/4.png
Saved: .output3/veo3_spectra_output/2.png
Saved: .output3/veo3_spectra_output/23.png
Saved: .output3/veo3_spectra_output/30.png
Saved: .output3/veo3_spectra_output/26.png
Saved: .output3/veo3_spectra_output/6.png
Saved: .output3/veo3_spectra_output/17.png
Saved: .output3/veo3_spectra_output/3.png
Saved: .output3/veo3_spectra_output/11.png
Saved: .output3/veo3_spectra_output/16.png
Saved: .output3/veo3_spectra_output/22.png
Saved: .output3/veo3_spectra_output/28.png
Saved: .output3/veo3_spectra_output/21.png
Saved: .output3/veo3_spectra_output/29.png
Saved: .output3/v

In [ ]:
# FIXED VIDEO PROCESSING SCRIPT 
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
import imageio.v2 as imageio
from tqdm import tqdm
from torchaudio_filters import LowPass
from pathlib import Path

# ========== CONFIGURATION ==========
INPUT_FOLDER = "/home/aarushi/Detecting-Deepfakes/FaceForensics++/manipulated_sequences/DeepFakeDetection/frames"
OUTPUT_FOLDER = "/home/aarushi/Detecting-Deepfakes/fingerprints/FaceForensics++/DeepFakeDetection"
FRAME_LIMIT = None
FRAME_SIZE = 256
PATCH_SIZE = 32
BATCH_SIZE = 5000
DEVICE = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

# Butterworth Filter Parameters
CUTOFF = 10
FS = 100
ORDER = 2
# ===================================

# Initialize Low-Pass Filter
lpf = LowPass(cutoff=CUTOFF, sample_rate=FS, order=ORDER).to(DEVICE)
print(f"Using device: {DEVICE}")

def process_video(video_path, output_path):
    try:
        reader = imageio.get_reader(str(video_path))
        frames_list = []

        for frame_idx, frame in enumerate(reader):
            if FRAME_LIMIT and frame_idx >= FRAME_LIMIT:
                break
            try:
                img = Image.fromarray(frame).convert('L')
                img = img.resize((FRAME_SIZE, FRAME_SIZE), Image.LANCZOS)
                img_np = np.asarray(img, dtype=np.float32) / 255.0
                frames_list.append(img_np)
            except Exception as e:
                print(f"Skipping frame {frame_idx}: {e}")
                continue
        reader.close()

        if not frames_list:
            print(f"No valid frames for {video_path.name}")
            return

        # Convert to tensor on GPU
        all_frames = torch.from_numpy(np.stack(frames_list)).unsqueeze(1).to(DEVICE)  # (N, 1, 256, 256)

        # Extract patches
        patches_per_dim = FRAME_SIZE // PATCH_SIZE
        total_patches = patches_per_dim * patches_per_dim
        all_patches = all_frames.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
        all_patches = all_patches.contiguous().view(-1, PATCH_SIZE, PATCH_SIZE)  # (N * P, 32, 32)

        # Process in batches
        frame_spectra = []
        for i in range(0, all_patches.shape[0], BATCH_SIZE):
            batch = all_patches[i:i + BATCH_SIZE]
            batch = batch.unsqueeze(1)  # (B, 1, 32, 32)

            # Apply low-pass filtering
            lowpass = lpf(batch).squeeze(1)
            lowpass_t = lpf(lowpass.transpose(-1, -2).unsqueeze(1)).squeeze(1).transpose(-1, -2)

            # High-pass spectrum
            highpass = batch.squeeze(1) - lowpass_t
            highpass_fft = torch.fft.fft2(highpass)
            highpass_shifted = torch.fft.fftshift(highpass_fft, dim=(-2, -1))
            highpass_log = torch.log(1 + torch.abs(highpass_shifted))

            frame_spectra.append(highpass_log.cpu())

        all_spectra = torch.cat(frame_spectra, dim=0)
        all_spectra = all_spectra.view(len(frames_list), total_patches, PATCH_SIZE, PATCH_SIZE)
        frame_avg_spectra = torch.mean(all_spectra, dim=1)  # (frames, 32, 32)

        spectra = np.array([f.numpy() for f in frame_avg_spectra])
        avg_spectrum = np.mean(spectra, axis=0)

        # Save spectrum as grayscale PNG
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        plt.imsave(output_path, avg_spectrum, cmap='gray')

        print(f"Saved: {output_path.name}")

    except Exception as e:
        print(f"Error processing {video_path}: {e}")

def main():
    input_dir = Path(INPUT_FOLDER)
    output_dir = Path(OUTPUT_FOLDER)
    video_paths = list(input_dir.glob("*"))

    video_paths = [p for p in video_paths if p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}]
    print(f"Found {len(video_paths)} video(s) to process.")

    for video_path in tqdm(video_paths, desc="Processing videos"):
        output_path = output_dir / f"{video_path.stem}_fingerprint.png"
        process_video(video_path, output_path)

if __name__ == "__main__":
    main()

Using device: cuda:1
Found 0 video(s) to process.


Processing videos: 0it [00:00, ?it/s]


In [ ]:
# IMAGE PROCESSING SCRIPT

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from tqdm import tqdm
from torchaudio_filters import LowPass
from pathlib import Path

# ========== CONFIGURATION ==========
INPUT_FOLDER = "/home/aarushi/Detecting-Deepfakes/FaceForensics++/manipulated_sequences/DeepFakeDetection/c23/frames"
OUTPUT_FOLDER = "/home/aarushi/Detecting-Deepfakes/fingerprints/FaceForensics++/DeepFakeDetection"
FRAME_SIZE = 256
PATCH_SIZE = 32
BATCH_SIZE = 5000
DEVICE = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

# Butterworth Filter Parameters
CUTOFF = 10
FS = 100
ORDER = 2
# ===================================

# Initialize Low-Pass Filter
lpf = LowPass(cutoff=CUTOFF, sample_rate=FS, order=ORDER).to(DEVICE)
print(f"Using device: {DEVICE}")

def sanitize_path(p: Path) -> str:
    # Convert path to a safe string like: sub1_sub2_sub3
    return "_".join(p.parts)

def process_image_folder(folder_path, output_file_path):
    try:
        image_paths = sorted([p for p in folder_path.glob("*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png']])
        if not image_paths:
            print(f"No images found in {folder_path}")
            return

        frames_list = []
        for img_path in image_paths:
            try:
                img = Image.open(img_path).convert('L')
                img = img.resize((FRAME_SIZE, FRAME_SIZE), Image.LANCZOS)
                img_np = np.asarray(img, dtype=np.float32) / 255.0
                frames_list.append(img_np)
            except Exception as e:
                print(f"Skipping {img_path.name}: {e}")
                continue

        if not frames_list:
            print(f"No valid images in {folder_path}")
            return

        # Convert to tensor on GPU
        all_frames = torch.from_numpy(np.stack(frames_list)).unsqueeze(1).to(DEVICE)

        # Extract patches
        patches_per_dim = FRAME_SIZE // PATCH_SIZE
        total_patches = patches_per_dim * patches_per_dim
        all_patches = all_frames.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
        all_patches = all_patches.contiguous().view(-1, PATCH_SIZE, PATCH_SIZE)

        # Process in batches
        frame_spectra = []
        for i in range(0, all_patches.shape[0], BATCH_SIZE):
            batch = all_patches[i:i + BATCH_SIZE]
            batch = batch.unsqueeze(1)

            # Apply low-pass filtering
            lowpass = lpf(batch).squeeze(1)
            lowpass_t = lpf(lowpass.transpose(-1, -2).unsqueeze(1)).squeeze(1).transpose(-1, -2)

            highpass = batch.squeeze(1) - lowpass_t
            highpass_fft = torch.fft.fft2(highpass)
            highpass_shifted = torch.fft.fftshift(highpass_fft, dim=(-2, -1))
            highpass_log = torch.log(1 + torch.abs(highpass_shifted))

            frame_spectra.append(highpass_log.cpu())

        all_spectra = torch.cat(frame_spectra, dim=0)
        all_spectra = all_spectra.view(len(frames_list), total_patches, PATCH_SIZE, PATCH_SIZE)
        frame_avg_spectra = torch.mean(all_spectra, dim=1)

        spectra = np.array([f.numpy() for f in frame_avg_spectra])
        avg_spectrum = np.mean(spectra, axis=0)

        output_file_path.parent.mkdir(parents=True, exist_ok=True)
        plt.imsave(output_file_path, avg_spectrum, cmap='gray')
        print(f"Saved: {output_file_path.name}")

    except Exception as e:
        print(f"Error processing {folder_path}: {e}")

def main():
    input_root = Path(INPUT_FOLDER)
    output_root = Path(OUTPUT_FOLDER)

    subfolders = [f for f in input_root.rglob('*') if f.is_dir()]
    print(f"Found {len(subfolders)} subfolders with images.")

    output_root.mkdir(parents=True, exist_ok=True)

    for subfolder in tqdm(subfolders, desc="Processing image folders"):
        rel_path = subfolder.relative_to(input_root)
        flat_name = sanitize_path(rel_path) or subfolder.name
        output_file = output_root / f"{flat_name}_fingerprint.png"
        process_image_folder(subfolder, output_file)

if __name__ == "__main__":
    main()


Using device: cuda:1
Found 3066 subfolders with images.


Processing image folders:   0%|          | 2/3066 [00:00<14:20,  3.56it/s]

Saved: 27_26__meeting_serious__IE9N0ZI9_fingerprint.png
Saved: 12_03__walk_down_hall_angry__D7G6YJON_fingerprint.png


Processing image folders:   0%|          | 3/3066 [00:00<10:10,  5.02it/s]

Saved: 26_20__kitchen_pan__34DJQS3E_fingerprint.png
Saved: 21_09__podium_speech_happy__JEELOMN2_fingerprint.png


...

Processing image folders:  22%|██▏       | 672/3066 [02:10<05:51,  6.82it/s]

Saved: 13_26__talking_angry_couch__1ZRIYSSU_fingerprint.png
Saved: 11_06__talking_against_wall__C8C4FW99_fingerprint.png


Processing image folders:  22%|██▏       | 674/3066 [02:10<05:39,  7.04it/s]

Saved: 08_28__outside_talking_pan_laughing__HIOU8Z8I_fingerprint.png
Saved: 09_26__walking_down_street_outside_angry__C3K20JOL_fingerprint.png


Processing image folders:  22%|██▏       | 675/3066 [02:10<05:27,  7.30it/s]